## Creating a combined supervised Model best of both worlds

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score
)

In [ ]:
# Shared Data Prep

# Load features
features_df = pd.read_csv("Features_volatility.csv")
features_df["date"] = pd.to_datetime(features_df["date"])
features_df = features_df.sort_values("date").set_index("date")

# Lagged predictors
for i in range(1, 6):
    features_df[f"SPY_return_lag{i}"] = features_df["SPY_return"].shift(i)
    features_df[f"VIX_change_lag{i}"] = features_df["VIX_change"].shift(i)
    features_df[f"yield_spread_lag{i}"] = features_df["yield_spread"].shift(i)

# Optional: transition-sensitive features
features_df["SPY_vol_ratio"] = features_df["SPY_vol_4"] / features_df["SPY_vol_12"]
features_df["SPY_vol_spread"] = features_df["SPY_vol_4"] - features_df["SPY_vol_12"]
features_df["SPY_return_rolling_4"] = features_df["SPY_return"].rolling(4).mean()
features_df["QQQ_return_rolling_4"] = features_df["QQQ_return"].rolling(4).mean()

# Shared predictive feature set
predict_cols = [
    "SPY_return",
    "QQQ_return",
    "SPY_momentum_12",
    "VIX_change",
    "SPY_vol_4",
    "SPY_vol_12",
    "yield_spread",
    "SPY_return_lag1",
    "SPY_return_lag2",
    "SPY_return_lag3",
    "SPY_return_lag4",
    "SPY_return_lag5",
    "VIX_change_lag1",
    "VIX_change_lag2",
    "VIX_change_lag3",
    "VIX_change_lag4",
    "VIX_change_lag5",
    "yield_spread_lag1",
    "yield_spread_lag2",
    "yield_spread_lag3",
    "yield_spread_lag4",
    "yield_spread_lag5",
    "SPY_vol_ratio",
    "SPY_vol_spread",
    "SPY_return_rolling_4",
    "QQQ_return_rolling_4",
]

# Load regime labels
regime_df = pd.read_parquet("regime_labeled_k_combine.parquet").sort_index()

# Align datasets
common_idx = features_df.index.intersection(regime_df.index)
features_df = features_df.loc[common_idx].copy()
regime_df = regime_df.loc[common_idx].copy()

# Map regime names
regime_map = {
    0: "Low_Vol",
    1: "Mid_Vol",
    2: "High_Vol"
}
reg_cols = ["Low_Vol", "Mid_Vol", "High_Vol"]

regime_df["current_regime"] = regime_df["regime_3class"].map(regime_map)
regime_df["target_regime"] = regime_df["regime_3class"].shift(-1)
regime_df["next_regime"] = regime_df["target_regime"].map(regime_map)

# Updated Layer 1 target: risk vs calm
regime_df["target_risk"] = np.where(
    regime_df["next_regime"].isin(["Mid_Vol", "High_Vol"]),
    "risk",
    "calm"
)
regime_df.loc[regime_df["next_regime"].isna(), "target_risk"] = np.nan

# Retain the original change target for model comparison
regime_df["target_change"] = np.where(
    regime_df["next_regime"] != regime_df["current_regime"],
    "change",
    "no change"
)
regime_df.loc[regime_df["next_regime"].isna(), "target_change"] = np.nan

# Build final master modeling frame
model_df = features_df[predict_cols].copy()
model_df["current_regime"] = regime_df["current_regime"]
model_df["next_regime"] = regime_df["next_regime"]
model_df["target_risk"] = regime_df["target_risk"]
model_df["target_change"] = regime_df["target_change"]

model_df = model_df.dropna().copy()

print(f"Modeling rows: {len(model_df):,}")
print(f"Date range: {model_df.index.min().date()} to {model_df.index.max().date()}")
print()
print("Current regime counts:")
print(model_df["current_regime"].value_counts())
print()
print("Next regime counts:")
print(model_df["next_regime"].value_counts())
print()
print("Risk target counts:")
print(model_df["target_risk"].value_counts())
print()
print("Change target counts:")
print(model_df["target_change"].value_counts())
print()

In [ ]:
# Shared splits
TRAIN_END = "2016-12-31"
VAL_END   = "2019-12-31"


train = model_df.loc[:TRAIN_END].copy()
val = model_df.loc[(model_df.index > TRAIN_END) & (model_df.index <= VAL_END)].copy()
test = model_df.loc[model_df.index > VAL_END].copy()

print(f"Training rows:   {len(train):,} ({train.index.min().date()} to {train.index.max().date()})")
print(f"Validation rows: {len(val):,} ({val.index.min().date()} to {val.index.max().date()})")
print(f"Testing rows:    {len(test):,} ({test.index.min().date()} to {test.index.max().date()})")
print()

for split_name, df_ in [("Train", train), ("Validation", val), ("Test", test)]:

    print(f"{split_name} current_regime counts:")
    print(df_["current_regime"].value_counts())
    print()

    print(f"{split_name} next_regime counts:")
    print(df_["next_regime"].value_counts())
    print()
    
    print(f"{split_name} target_risk counts:")
    print(df_["target_risk"].value_counts())
    print()

In [ ]:
# Baselines

# Direct multiclass persistence
persist_val_reg = val["current_regime"].copy()
persist_test_reg = test["current_regime"].copy()

# Majority baseline for direct next-regime model
majority_direct = train["next_regime"].value_counts().idxmax()
majority_direct_val = pd.Series(majority_direct, index=val.index)
majority_direct_test = pd.Series(majority_direct, index=test.index)

# Majority baseline for Layer 1 risk model
majority_risk = train["target_risk"].value_counts().idxmax()
majority_risk_val = pd.Series(majority_risk, index=val.index)
majority_risk_test = pd.Series(majority_risk, index=test.index)

print("Direct majority baseline regime:", majority_direct)
print("Layer 1 majority baseline:", majority_risk)
print()

In [ ]:
# Arun model - direct next regime prediction

# Random Forest
rf_arun = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf_arun.fit(X_train_arun, y_train_arun)

arun_rf_val_pred = rf_arun.predict(X_val_arun)
arun_rf_test_pred = rf_arun.predict(X_test_arun)

print("ARUN RANDOM FOREST - Validation")
print(classification_report(y_val_arun, arun_rf_val_pred, labels=reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_arun, arun_rf_val_pred), 4))
print()

print("ARUN RANDOM FOREST - Test")
print(classification_report(y_test_arun, arun_rf_test_pred, labels=reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_arun, arun_rf_test_pred), 4))
print()

In [ ]:
# Phuc Model 2 Layer Approach

print("=" * 75)
print("PHUC Model — 2-layer Approach")
print("=" * 75)

# Layer 1: risk / calm
X_train_l1 = train[predict_cols]
X_val_l1 = val[predict_cols]
X_test_l1 = test[predict_cols]

y_train_l1 = train["target_risk"]
y_val_l1 = val["target_risk"]
y_test_l1 = test["target_risk"]

scaler_l1 = StandardScaler()
X_train_l1_sc = scaler_l1.fit_transform(X_train_l1)
X_val_l1_sc = scaler_l1.transform(X_val_l1)
X_test_l1_sc = scaler_l1.transform(X_test_l1)

# Logistic Regression for Layer 1
lr_l1 = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)
lr_l1.fit(X_train_l1_sc, y_train_l1)

risk_idx = list(lr_l1.classes_).index("risk")
p_val_l1 = lr_l1.predict_proba(X_val_l1_sc)[:, risk_idx]
p_test_l1 = lr_l1.predict_proba(X_test_l1_sc)[:, risk_idx]

# Threshold selection:
# prioritize catching risk while avoiding a completely crazy alert rate
thresholds = np.arange(0.05, 0.51, 0.01)
best_threshold = 0.50
best_score = -1

for t in thresholds:
    temp_pred = np.where(p_val_l1 >= t, "risk", "calm")
    pred_rate = (temp_pred == "risk").mean()

    rec = recall_score(y_val_l1, temp_pred, pos_label="risk", zero_division=0)
    prec = precision_score(y_val_l1, temp_pred, pos_label="risk", zero_division=0)

    # weighted score: favor recall because missing risk is worse than false alarms
    score = 0.7 * rec + 0.3 * prec

    # require model to actually fire sometimes, but not all the time
    if 0.08 <= pred_rate <= 0.50 and score > best_score:
        best_score = score
        best_threshold = t

phuc_l1_lr_val_pred = np.where(p_val_l1 >= best_threshold, "risk", "calm")
phuc_l1_lr_test_pred = np.where(p_test_l1 >= best_threshold, "risk", "calm")

print("PHUC LAYER 1 LOGISTIC - Validation")
print(classification_report(y_val_l1, phuc_l1_lr_val_pred, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_l1, phuc_l1_lr_val_pred), 4))
print("Validation predicted risk rate:", round((phuc_l1_lr_val_pred == 'risk').mean(), 4))
print()

print("PHUC LAYER 1 LOGISTIC - Test")
print(classification_report(y_test_l1, phuc_l1_lr_test_pred, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_l1, phuc_l1_lr_test_pred), 4))
print("Test predicted risk rate:", round((phuc_l1_lr_test_pred == 'risk').mean(), 4))
print()

print(f"Best Layer 1 LR threshold: {best_threshold:.2f}")
print(f"Best Layer 1 validation score: {best_score:.4f}")
print()

In [ ]:
# Choosing Arun best model by validation accuracy
arun_lr_val_acc = accuracy_score(y_val_arun, arun_lr_val_pred)
arun_rf_val_acc = accuracy_score(y_val_arun, arun_rf_val_pred)

if arun_lr_val_acc >= arun_rf_val_acc:
    arun_best_name = "Logistic Regression"
    arun_val_pred = pd.Series(arun_lr_val_pred, index=val.index)
    arun_test_pred = pd.Series(arun_lr_test_pred, index=test.index)
    arun_best_model = lr_arun
else:
    arun_best_name = "Random Forest"
    arun_val_pred = pd.Series(arun_rf_val_pred, index=val.index)
    arun_test_pred = pd.Series(arun_rf_test_pred, index=test.index)
    arun_best_model = rf_arun

print("Arun model selection:")
print(f"  Logistic Regression val accuracy: {arun_lr_val_acc:.4f}")
print(f"  Random Forest val accuracy:       {arun_rf_val_acc:.4f}")
print(f"Selected Arun model: {arun_best_name}")
print()

In [ ]:
# Arun model - direct next regime prediction
print("=" * 75)
print("ARUN MODEL - Direct multiclass next regime prediction")
print("=" * 75)

X_train_arun = train[predict_cols]
X_val_arun = val[predict_cols]
X_test_arun = test[predict_cols]

y_train_arun = train["next_regime"]
y_val_arun = val["next_regime"]
y_test_arun = test["next_regime"]

# Scaling for logistic regression
scaler_arun = StandardScaler()
X_train_arun_sc = scaler_arun.fit_transform(X_train_arun)
X_val_arun_sc = scaler_arun.transform(X_val_arun)
X_test_arun_sc = scaler_arun.transform(X_test_arun)

# Logistic Regression
lr_arun = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)
lr_arun.fit(X_train_arun_sc, y_train_arun)

arun_lr_val_pred = lr_arun.predict(X_val_arun_sc)
arun_lr_test_pred = lr_arun.predict(X_test_arun_sc)

print("ARUN LOGISTIC REGRESSION - Validation")
print(classification_report(y_val_arun, arun_lr_val_pred, labels=reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_arun, arun_lr_val_pred), 4))
print()

print("ARUN LOGISTIC REGRESSION - Test")
print(classification_report(y_test_arun, arun_lr_test_pred, labels=reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_arun, arun_lr_test_pred), 4))
print()

In [ ]:
# Random Forest for Layer 1

rf_l1 = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf_l1.fit(X_train_l1, y_train_l1)

risk_idx_rf = list(rf_l1.classes_).index("risk")
p_val_l1_rf = rf_l1.predict_proba(X_val_l1)[:, risk_idx_rf]
p_test_l1_rf = rf_l1.predict_proba(X_test_l1)[:, risk_idx_rf]

best_threshold_rf = 0.50
best_score_rf = -1

for t in np.arange(0.05, 0.51, 0.01):
    temp_pred = np.where(p_val_l1_rf >= t, "risk", "calm")
    pred_rate = (temp_pred == "risk").mean()

    rec = recall_score(y_val_l1, temp_pred, pos_label="risk", zero_division=0)
    prec = precision_score(y_val_l1, temp_pred, pos_label="risk", zero_division=0)

    score = 0.7 * rec + 0.3 * prec

    if 0.08 <= pred_rate <= 0.50 and score > best_score_rf:
        best_score_rf = score
        best_threshold_rf = t

phuc_l1_rf_val_pred = np.where(p_val_l1_rf >= best_threshold_rf, "risk", "calm")
phuc_l1_rf_test_pred = np.where(p_test_l1_rf >= best_threshold_rf, "risk", "calm")

print("PHUC LAYER 1 RANDOM FOREST - Validation")
print(classification_report(y_val_l1, phuc_l1_rf_val_pred, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_l1, phuc_l1_rf_val_pred), 4))
print("Validation predicted risk rate:", round((phuc_l1_rf_val_pred == 'risk').mean(), 4))
print()

print("PHUC LAYER 1 RANDOM FOREST - Test")
print(classification_report(y_test_l1, phuc_l1_rf_test_pred, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_l1, phuc_l1_rf_test_pred), 4))
print("Test predicted risk rate:", round((phuc_l1_rf_test_pred == 'risk').mean(), 4))
print()

print(f"Best Layer 1 RF threshold: {best_threshold_rf:.2f}")
print(f"Best Layer 1 RF validation score: {best_score_rf:.4f}")
print()

In [ ]:
# Pick best Layer 1 model using recall-sensitive validation score for "risk"
phuc_l1_lr_val_rec = recall_score(y_val_l1, phuc_l1_lr_val_pred, pos_label="risk", zero_division=0)
phuc_l1_lr_val_prec = precision_score(y_val_l1, phuc_l1_lr_val_pred, pos_label="risk", zero_division=0)
phuc_l1_lr_val_score = 0.7 * phuc_l1_lr_val_rec + 0.3 * phuc_l1_lr_val_prec

phuc_l1_rf_val_rec = recall_score(y_val_l1, phuc_l1_rf_val_pred, pos_label="risk", zero_division=0)
phuc_l1_rf_val_prec = precision_score(y_val_l1, phuc_l1_rf_val_pred, pos_label="risk", zero_division=0)
phuc_l1_rf_val_score = 0.7 * phuc_l1_rf_val_rec + 0.3 * phuc_l1_rf_val_prec

if phuc_l1_lr_val_score >= phuc_l1_rf_val_score:
    phuc_l1_name = "Logistic Regression"
    l1_val_pred = pd.Series(phuc_l1_lr_val_pred, index=val.index)
    l1_test_pred = pd.Series(phuc_l1_lr_test_pred, index=test.index)
else:
    phuc_l1_name = "Random Forest"
    l1_val_pred = pd.Series(phuc_l1_rf_val_pred, index=val.index)
    l1_test_pred = pd.Series(phuc_l1_rf_test_pred, index=test.index)

print("Phuc Layer 1 model selection:")
print(f"  Logistic val recall (risk):   {phuc_l1_lr_val_rec:.4f}")
print(f"  Logistic val precision (risk): {phuc_l1_lr_val_prec:.4f}")
print(f"  Logistic val score:            {phuc_l1_lr_val_score:.4f}")
print()
print(f"  RF val recall (risk):          {phuc_l1_rf_val_rec:.4f}")
print(f"  RF val precision (risk):       {phuc_l1_rf_val_prec:.4f}")
print(f"  RF val score:                  {phuc_l1_rf_val_score:.4f}")
print()
print(f"Selected Phuc Layer 1 model: {phuc_l1_name}")
print()

In [ ]:
# Layer 2

predict_cols_l2 = [
    "SPY_return",
    "QQQ_return",
    "VIX_change",
    "SPY_vol_4",
    "yield_spread",
    "SPY_return_lag1",
    "SPY_return_lag2",
    "VIX_change_lag1",
    "VIX_change_lag2",
    "yield_spread_lag1",
    "yield_spread_lag2",
    "SPY_vol_ratio",
    "SPY_vol_spread",
    "SPY_return_rolling_4",
    "QQQ_return_rolling_4",
]

# Layer 2 now works on the rows where next regime is risky
risk_df = model_df[model_df["target_risk"] == "risk"].copy()

l2_train = risk_df.loc[:TRAIN_END].copy()
l2_val = risk_df.loc[(risk_df.index > TRAIN_END) & (risk_df.index <= VAL_END)].copy()
l2_test = risk_df.loc[risk_df.index > VAL_END].copy()

# Layer 2 target is now only Mid_Vol vs High_Vol
y_l2_train = l2_train["next_regime"]
y_l2_val = l2_val["next_regime"]
y_l2_test = l2_test["next_regime"]

print("Layer 2 train target counts:")
print(y_l2_train.value_counts())
print()
print("Layer 2 val target counts:")
print(y_l2_val.value_counts())
print()
print("Layer 2 test target counts:")
print(y_l2_test.value_counts())
print()

# Conditional baseline for layer 2
transition_table = pd.crosstab(l2_train["current_regime"], l2_train["next_regime"])
most_likely_next = transition_table.idxmax(axis=1)
fallback_regime = y_l2_train.value_counts().idxmax()

l2_base_val = l2_val["current_regime"].map(most_likely_next).fillna(fallback_regime)
l2_base_test = l2_test["current_regime"].map(most_likely_next).fillna(fallback_regime)

# Build l2 design matrices with current regime dummies
def build_l2_features(df, fit_columns=None):
    x = df[predict_cols_l2].copy()
    d = pd.get_dummies(df["current_regime"], prefix="cur")
    x = pd.concat([x, d], axis=1)
    if fit_columns is not None:
        x = x.reindex(columns=fit_columns, fill_value=0)
    return x

X_l2_train = build_l2_features(l2_train)
X_l2_val = build_l2_features(l2_val, fit_columns=X_l2_train.columns)
X_l2_test = build_l2_features(l2_test, fit_columns=X_l2_train.columns)

# Scale numeric cols only for LR
scaler_l2 = StandardScaler()

X_l2_train_sc = X_l2_train.copy()
X_l2_val_sc = X_l2_val.copy()
X_l2_test_sc = X_l2_test.copy()

X_l2_train_sc[predict_cols_l2] = scaler_l2.fit_transform(X_l2_train[predict_cols_l2])
X_l2_val_sc[predict_cols_l2] = scaler_l2.transform(X_l2_val[predict_cols_l2])
X_l2_test_sc[predict_cols_l2] = scaler_l2.transform(X_l2_test[predict_cols_l2])

# Layer 2 Logistic
lr_l2 = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)
lr_l2.fit(X_l2_train_sc, y_l2_train)

phuc_l2_lr_val_pred = lr_l2.predict(X_l2_val_sc)
phuc_l2_lr_test_pred = lr_l2.predict(X_l2_test_sc)

print("PHUC LAYER 2 LOGISTIC - Validation")
print(classification_report(y_l2_val, phuc_l2_lr_val_pred, labels=["Mid_Vol", "High_Vol"], digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_l2_val, phuc_l2_lr_val_pred), 4))
print()

print("PHUC LAYER 2 LOGISTIC - Test")
print(classification_report(y_l2_test, phuc_l2_lr_test_pred, labels=["Mid_Vol", "High_Vol"], digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_l2_test, phuc_l2_lr_test_pred), 4))
print()

In [ ]:
rows = []

rows.append({
    "Model": "Persistence Baseline",
    "Validation Accuracy": round(accuracy_score(y_val_arun, persist_val_reg), 4),
    "Test Accuracy": round(accuracy_score(y_test_arun, persist_test_reg), 4),
    "Test Precision (weighted)": round(
        precision_score(y_test_arun, persist_test_reg, average="weighted", zero_division=0), 4
    ),
    "Test Recall (weighted)": round(
        recall_score(y_test_arun, persist_test_reg, average="weighted", zero_division=0), 4
    ),
    "Test F1 (macro)": round(
        f1_score(y_test_arun, persist_test_reg, average="macro", zero_division=0), 4
    ),
})

rows.append({
    "Model": "Arun Direct Model",
    "Validation Accuracy": round(accuracy_score(y_val_arun, arun_val_pred), 4),
    "Test Accuracy": round(accuracy_score(y_test_arun, arun_test_pred), 4),
    "Test Precision (weighted)": round(
        precision_score(y_test_arun, arun_test_pred, average="weighted", zero_division=0), 4
    ),
    "Test Recall (weighted)": round(
        recall_score(y_test_arun, arun_test_pred, average="weighted", zero_division=0), 4
    ),
    "Test F1 (macro)": round(
        f1_score(y_test_arun, arun_test_pred, average="macro", zero_division=0), 4
    ),
})

rows.append({
    "Model": "Phuc 2-Layer Baseline",
    "Validation Accuracy": round(accuracy_score(y_val_arun, phuc_base_val_pred), 4),
    "Test Accuracy": round(accuracy_score(y_test_arun, phuc_base_test_pred), 4),
    "Test Precision (weighted)": round(
        precision_score(y_test_arun, phuc_base_test_pred, average="weighted", zero_division=0), 4
    ),
    "Test Recall (weighted)": round(
        recall_score(y_test_arun, phuc_base_test_pred, average="weighted", zero_division=0), 4
    ),
    "Test F1 (macro)": round(
        f1_score(y_test_arun, phuc_base_test_pred, average="macro", zero_division=0), 4
    ),
})

rows.append({
    "Model": "Phuc 2-Layer Model",
    "Validation Accuracy": round(accuracy_score(y_val_arun, phuc_val_pred), 4),
    "Test Accuracy": round(accuracy_score(y_test_arun, phuc_test_pred), 4),
    "Test Precision (weighted)": round(
        precision_score(y_test_arun, phuc_test_pred, average="weighted", zero_division=0), 4
    ),
    "Test Recall (weighted)": round(
        recall_score(y_test_arun, phuc_test_pred, average="weighted", zero_division=0), 4
    ),
    "Test F1 (macro)": round(
        f1_score(y_test_arun, phuc_test_pred, average="macro", zero_division=0), 4
    ),
})

summary_df = pd.DataFrame(rows).set_index("Model")

print("FINAL MODEL COMPARISON")
print(summary_df.to_string())
print()

In [ ]:
# Layer 2 Random Forest
rf_l2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf_l2.fit(X_l2_train, y_l2_train)

phuc_l2_rf_val_pred = rf_l2.predict(X_l2_val)
phuc_l2_rf_test_pred = rf_l2.predict(X_l2_test)

l2_reg_cols = ["Mid_Vol", "High_Vol"]

print("PHUC LAYER 2 RANDOM FOREST - Validation")
print(classification_report(y_l2_val, phuc_l2_rf_val_pred, labels=l2_reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_l2_val, phuc_l2_rf_val_pred), 4))
print()

print("PHUC LAYER 2 RANDOM FOREST - Test")
print(classification_report(y_l2_test, phuc_l2_rf_test_pred, labels=l2_reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_l2_test, phuc_l2_rf_test_pred), 4))
print()

print("PHUC LAYER 2 CONDITIONAL BASELINE - Validation")
print(classification_report(y_l2_val, l2_base_val, labels=l2_reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_l2_val, l2_base_val), 4))
print()

print("PHUC LAYER 2 CONDITIONAL BASELINE - Test")
print(classification_report(y_l2_test, l2_base_test, labels=l2_reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_l2_test, l2_base_test), 4))
print()

In [ ]:
# Combining both layers

X_l2_val_all = build_l2_features(val, fit_columns=X_l2_train.columns)
X_l2_test_all = build_l2_features(test, fit_columns=X_l2_train.columns)

X_l2_val_all_sc = X_l2_val_all.copy()
X_l2_test_all_sc = X_l2_test_all.copy()

X_l2_val_all_sc[predict_cols_l2] = scaler_l2.transform(X_l2_val_all[predict_cols_l2])
X_l2_test_all_sc[predict_cols_l2] = scaler_l2.transform(X_l2_test_all[predict_cols_l2])

# Best Layer 2 model predictions across all validation/test rows
if phuc_l2_best_is_lr:
    l2_val_all_pred = pd.Series(lr_l2.predict(X_l2_val_all_sc), index=val.index)
    l2_test_all_pred = pd.Series(lr_l2.predict(X_l2_test_all_sc), index=test.index)
else:
    l2_val_all_pred = pd.Series(rf_l2.predict(X_l2_val_all), index=val.index)
    l2_test_all_pred = pd.Series(rf_l2.predict(X_l2_test_all), index=test.index)

# Layer 2 baseline predictions across all rows
l2_base_val_all = val["current_regime"].map(most_likely_next).fillna(fallback_regime)
l2_base_test_all = test["current_regime"].map(most_likely_next).fillna(fallback_regime)

# Final combined 2-layer prediction
# Default = Low_Vol when Layer 1 says calm
# If Layer 1 says risk, use Layer 2 prediction (Mid_Vol / High_Vol)
phuc_val_pred = pd.Series("Low_Vol", index=val.index)
phuc_test_pred = pd.Series("Low_Vol", index=test.index)

phuc_val_pred[l1_val_pred == "risk"] = l2_val_all_pred[l1_val_pred == "risk"]
phuc_test_pred[l1_test_pred == "risk"] = l2_test_all_pred[l1_test_pred == "risk"]

# Final baseline version
phuc_base_val_pred = pd.Series("Low_Vol", index=val.index)
phuc_base_test_pred = pd.Series("Low_Vol", index=test.index)

phuc_base_val_pred[l1_val_pred == "risk"] = l2_base_val_all[l1_val_pred == "risk"]
phuc_base_test_pred[l1_test_pred == "risk"] = l2_base_test_all[l1_test_pred == "risk"]

print("FULL PHUC 2-LAYER MODEL - Validation")
print(classification_report(y_val_arun, phuc_val_pred, labels=reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_arun, phuc_val_pred), 4))
print()

print("FULL PHUC 2-LAYER MODEL - Test")
print(classification_report(y_test_arun, phuc_test_pred, labels=reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_arun, phuc_test_pred), 4))
print()

print("FULL PHUC 2-LAYER BASELINE - Validation")
print(classification_report(y_val_arun, phuc_base_val_pred, labels=reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_arun, phuc_base_val_pred), 4))
print()

print("FULL PHUC 2-LAYER BASELINE - Test")
print(classification_report(y_test_arun, phuc_base_test_pred, labels=reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_arun, phuc_base_test_pred), 4))
print()

print("FULL PERSISTENCE BASELINE - Validation")
print(classification_report(y_val_arun, persist_val_reg, labels=reg_cols, digits=3, zero_division=0))
print("Validation accuracy:", round(accuracy_score(y_val_arun, persist_val_reg), 4))
print()

print("FULL PERSISTENCE BASELINE - Test")
print(classification_report(y_test_arun, persist_test_reg, labels=reg_cols, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_test_arun, persist_test_reg), 4))
print()

In [ ]:
# Pick best layer 2 model by validation accuracy
phuc_l2_lr_val_acc = accuracy_score(y_l2_val, phuc_l2_lr_val_pred)
phuc_l2_rf_val_acc = accuracy_score(y_l2_val, phuc_l2_rf_val_pred)

if phuc_l2_lr_val_acc >= phuc_l2_rf_val_acc:
    phuc_l2_name = "Logistic Regression"
    phuc_l2_best_is_lr = True
else:
    phuc_l2_name = "Random Forest"
    phuc_l2_best_is_lr = False

print("Phuc Layer 2 model selection:")
print(f"  Logistic val accuracy: {phuc_l2_lr_val_acc:.4f}")
print(f"  RF val accuracy:       {phuc_l2_rf_val_acc:.4f}")
print(f"Selected Phuc Layer 2 model: {phuc_l2_name}")
print()

In [ ]:
# Confusion Matrices

models_to_plot = [
    ("Persistence", persist_test_reg),
    (f"Arun Direct ({arun_best_name})", arun_test_pred),
    ("Phuc 2-Layer", phuc_test_pred),
]

fig, axes = plt.subplots(1, len(models_to_plot), figsize=(6 * len(models_to_plot), 5))
if len(models_to_plot) == 1:
    axes = [axes]

for ax, (name, preds) in zip(axes, models_to_plot):
    cm = confusion_matrix(y_test_arun, preds, labels=reg_cols)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=reg_cols)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=12)
    ax.tick_params(axis="x", rotation=25)

plt.suptitle("Test Set Confusion Matrices", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
'''# Save predictions for portfolio notebooks

pred_export = test.copy()

# Actuals
pred_export["actual_next_regime"] = y_test_arun

# Model outputs
pred_export["persistence_pred"] = persist_test_reg
pred_export["arun_pred"] = arun_test_pred
pred_export["phuc_2layer_pred"] = phuc_test_pred

# Layer-level outputs (VERY useful)
pred_export["phuc_l1_pred"] = l1_test_pred
pred_export["phuc_l2_pred"] = l2_test_all_pred

# Optional: Layer 1 probability (if using LR as best)
try:
    pred_export["phuc_l1_risk_prob"] = p_test_l1
except:
    pass

pred_export.to_parquet("master_supervised_test_predictions.parquet")

print("Saved: master_supervised_test_predictions.parquet")'''

In [ ]:
# Save predictions for portfolio notebooks
# Export BOTH validation and test so downstream portfolio notebooks
# can learn allocation on validation and evaluate on test.

# Choose the correct Layer 1 probability series based on selected model
if phuc_l1_name == "Logistic Regression":
    l1_val_risk_prob = pd.Series(p_val_l1, index=val.index)
    l1_test_risk_prob = pd.Series(p_test_l1, index=test.index)
else:
    l1_val_risk_prob = pd.Series(p_val_l1_rf, index=val.index)
    l1_test_risk_prob = pd.Series(p_test_l1_rf, index=test.index)

# Validation export
pred_export_val = val.copy()
pred_export_val["dataset_split"] = "validation"

# Actuals
pred_export_val["actual_next_regime"] = y_val_arun

# Model outputs
pred_export_val["persistence_pred"] = persist_val_reg
pred_export_val["arun_pred"] = arun_val_pred
pred_export_val["phuc_2layer_pred"] = phuc_val_pred
pred_export_val["phuc_2layer_base_pred"] = phuc_base_val_pred

# Layer-level outputs
pred_export_val["phuc_l1_pred"] = l1_val_pred
pred_export_val["phuc_l2_pred"] = l2_val_all_pred
pred_export_val["phuc_l1_risk_prob"] = l1_val_risk_prob

# Test export
pred_export_test = test.copy()
pred_export_test["dataset_split"] = "test"

# Actuals
pred_export_test["actual_next_regime"] = y_test_arun

# Model outputs
pred_export_test["persistence_pred"] = persist_test_reg
pred_export_test["arun_pred"] = arun_test_pred
pred_export_test["phuc_2layer_pred"] = phuc_test_pred
pred_export_test["phuc_2layer_base_pred"] = phuc_base_test_pred

# Layer-level outputs
pred_export_test["phuc_l1_pred"] = l1_test_pred
pred_export_test["phuc_l2_pred"] = l2_test_all_pred
pred_export_test["phuc_l1_risk_prob"] = l1_test_risk_prob

# Combine and save
pred_export = pd.concat([pred_export_val, pred_export_test], axis=0).sort_index()

pred_export.to_parquet("master_supervised_predictions.parquet")

print("Saved: master_supervised_predictions.parquet")
print("\nRows by split:")
print(pred_export["dataset_split"].value_counts())

print("\nDate range:")
print(pred_export.index.min(), "to", pred_export.index.max())

print("\nColumns:")
print(pred_export.columns.tolist())

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=059a5e93-4459-411b-9a58-2a578fd7a892' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>